# Offline RL on CartPole: BC vs Naive-DQN vs CQL vs IQL

**A minimal, CPU-only, visual tour of offline reinforcement learning.**

In *offline* (a.k.a. *batch*) RL we are handed a **fixed dataset** of transitions
$\mathcal{D} = \{(s, a, r, s')\}$ collected by some unknown *behavior policy*, and we
must learn the best policy we can **without ever touching the environment during
learning**. The environment appears only twice in this notebook:

1. **Once**, up front, to *collect* the fixed dataset (Phase 1).
2. **At the end**, to *evaluate* the learned policies by rolling them out (Phase 2 eval).

No learning algorithm below ever calls `env.step()`. That is the whole point.

---

## The central problem: distributional shift & Q-value overestimation

Standard off-policy RL (e.g. DQN) bootstraps its target from
$\max_{a'} Q(s', a')$. When you can *interact*, an over-optimistic $Q$ for some
action gets corrected: the agent tries that action, sees the real (bad) reward,
and the estimate comes down. **Offline, that correction loop is severed.** The
Bellman target queries $Q$ at actions $a'$ that may be **out-of-distribution
(OOD)** — actions the behavior policy rarely or never took at $s'$. The network
is free to *hallucinate* high values there, those errors get bootstrapped into
targets, and $Q$ blows up. The greedy policy then chases these phantom values and
performs terribly in the real environment.

This is **distributional shift**: the learned policy induces a state-action
distribution different from the data, and the value estimates are only trustworthy
*on the data*.

### Four responses to this problem

| Method | One-line idea | Touches OOD actions? |
|---|---|---|
| **BC** (behavior cloning) | Just imitate the data actions (supervised). | Never queries $Q$ at all. |
| **Naive offline DQN** | Plain DQN on the buffer. | Yes — bootstraps $\max_{a'}Q$. **Our cautionary tale.** |
| **CQL** | DQN **+ a conservative penalty** that pushes $Q$ *down* on all actions and *up* on data actions. | Penalizes OOD over-estimation. |
| **IQL** | Never query $Q$ on OOD actions: fit $V$ by **expectile regression**, $Q\!\to\!r+\gamma V(s')$, extract policy by **advantage-weighted BC**. | Avoids OOD queries entirely. |

We will literally **plot** naive-DQN's overestimation against CQL's calibrated $Q$,
versus the *true* Monte-Carlo discounted returns in the dataset.

> **Connections.** The offline-RL "stay close to the data" principle is exactly
> the regularization story in **LLM offline post-training**: SFT is behavior
> cloning; DPO/RLHF add a KL-to-reference term that, like CQL's penalty, prevents
> the policy from drifting onto OOD (reward-hacking) completions. The same caution
> underlies **constrained scientific design** (molecules / sequences / proteins):
> a learned scorer is only reliable near the training distribution, so optimizing
> it naively yields adversarial, unrealizable designs — conservatism is the fix.
> See the companion Notion page:
> [Offline RL](https://app.notion.com/p/37f95008d766817283ffd9640fdb4c37).

## 0. Setup

Tiny install (only `gymnasium` for the env; `pygame` is its rendering backend).
Everything else (PyTorch, NumPy, Matplotlib) is preinstalled on Colab.

In [ ]:
%pip install -q "gymnasium>=0.29" "pygame>=2.5"

## 1. Imports, config & reproducibility

Everything is sized for **CPU**: small MLPs, ~50k transitions, a few thousand
gradient steps per method. The whole notebook runs in a couple of minutes.

In [ ]:
import time, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym

NB_T0 = time.time()  # global wall-clock start; we print elapsed time as we go.

# ----------------------------- CONFIG ---------------------------------------
# (All small / CPU-friendly. The smoke test overrides these to be even smaller.)
class CFG:
    ENV_ID        = "CartPole-v1"
    GAMMA         = 0.99          # discount
    # ---- Phase 1: behavior policy + dataset ----
    BEH_STEPS     = 15_000        # env steps to train the "medium" behavior DQN
    DATASET_SIZE  = 50_000        # number of transitions in the FIXED dataset
    EPS_COLLECT   = 0.30          # epsilon-random mixed into data collection (coverage)
    # ---- Phase 2: offline learning ----
    HIDDEN        = 128           # MLP hidden width
    BATCH         = 256
    OFFLINE_STEPS = 4_000         # gradient steps per offline method
    LR            = 1e-3
    TARGET_TAU    = 0.01          # soft target update rate (Polyak)
    # CQL
    CQL_ALPHA     = 1.0           # weight on the conservative penalty
    # IQL
    IQL_EXPECTILE = 0.8           # tau for upper-expectile V regression (0.7-0.9)
    IQL_BETA      = 3.0           # inverse-temperature for advantage weighting
    IQL_ADV_CLIP  = 100.0         # clip on exp(beta*adv) for numerical safety
    # ---- eval ----
    EVAL_EPISODES = 20            # rollouts per seed when evaluating a policy
    SEEDS         = [0, 1, 2]     # seeds for the offline learners + eval
    DEVICE        = "cpu"

cfg = CFG()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

set_seed(0)
print("torch", torch.__version__, "| gymnasium", gym.__version__, "| device", cfg.DEVICE)

## 2. Networks

Three tiny MLP heads, all sharing the same 2-hidden-layer trunk shape:

* **Q-network** `QNet`: $s \mapsto Q(s,\cdot) \in \mathbb{R}^{|A|}$ (used by DQN, CQL, IQL).
* **V-network** `VNet`: $s \mapsto V(s) \in \mathbb{R}$ (used by IQL only).
* **Policy head** `PolicyNet`: $s \mapsto \text{logits over } A$ (used by BC and IQL's actor).

`obs_dim=4`, `n_act=2` for CartPole — but we read them from the env so the code is generic.

In [ ]:
def mlp(in_dim, out_dim, hidden):
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, out_dim),
    )

class QNet(nn.Module):
    "Q(s, .) -> one value per discrete action."
    def __init__(self, obs_dim, n_act, hidden):
        super().__init__(); self.net = mlp(obs_dim, n_act, hidden)
    def forward(self, s): return self.net(s)

class VNet(nn.Module):
    "V(s) -> scalar."
    def __init__(self, obs_dim, hidden):
        super().__init__(); self.net = mlp(obs_dim, 1, hidden)
    def forward(self, s): return self.net(s).squeeze(-1)

class PolicyNet(nn.Module):
    "pi(.|s) as logits over discrete actions."
    def __init__(self, obs_dim, n_act, hidden):
        super().__init__(); self.net = mlp(obs_dim, n_act, hidden)
    def forward(self, s): return self.net(s)          # logits
    @torch.no_grad()
    def act(self, s_np):                               # greedy action from a numpy obs
        s = torch.as_tensor(s_np, dtype=torch.float32).unsqueeze(0)
        return int(self.forward(s).argmax(dim=-1).item())

## 3. Phase 1 — collect the FIXED dataset

We first train a **behavior policy** *partway* to "medium" skill with a quick
online DQN (this is the **only** place the environment is used for learning, and
it is *not* one of the four methods we compare — it just manufactures data). Then
we roll it out with $\epsilon$-greedy exploration ($\epsilon=0.30$) to collect
**~50k transitions** with decent state-action *coverage* but mediocre returns.

After this cell the dataset is **frozen** — stored as flat NumPy arrays in a
`Dataset` object. No method past this point sees the env (until eval).

In [ ]:
# ---- A self-contained, minimal online DQN to make a MEDIUM behavior policy. ----
# This is throwaway plumbing for data generation; the four *offline* learners
# we actually study come later and never call env.step().

def train_behavior_policy(cfg, seed=0):
    set_seed(seed)
    env = gym.make(cfg.ENV_ID)
    obs_dim = env.observation_space.shape[0]
    n_act   = env.action_space.n
    q   = QNet(obs_dim, n_act, cfg.HIDDEN)
    qt  = QNet(obs_dim, n_act, cfg.HIDDEN); qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=cfg.LR)
    buf = []  # simple replay list of (s,a,r,s2,done)
    s, _ = env.reset(seed=seed)
    for t in range(cfg.BEH_STEPS):
        eps = max(0.05, 1.0 - t / (0.6 * cfg.BEH_STEPS))   # decay exploration
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = int(q(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
        s2, r, term, trunc, _ = env.step(a)
        buf.append((s, a, r, s2, float(term)))
        if len(buf) > 20_000: buf.pop(0)
        s = s2 if not (term or trunc) else env.reset()[0]
        # --- DQN update ---
        if len(buf) >= cfg.BATCH:
            idx = np.random.randint(0, len(buf), size=cfg.BATCH)
            S  = torch.as_tensor(np.array([buf[i][0] for i in idx]), dtype=torch.float32)
            A  = torch.as_tensor([buf[i][1] for i in idx], dtype=torch.long)
            R  = torch.as_tensor([buf[i][2] for i in idx], dtype=torch.float32)
            S2 = torch.as_tensor(np.array([buf[i][3] for i in idx]), dtype=torch.float32)
            D  = torch.as_tensor([buf[i][4] for i in idx], dtype=torch.float32)
            with torch.no_grad():
                tgt = R + cfg.GAMMA * (1 - D) * qt(S2).max(dim=1).values
            qsa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)
            loss = F.smooth_l1_loss(qsa, tgt)
            opt.zero_grad(); loss.backward(); opt.step()
            for p, pt in zip(q.parameters(), qt.parameters()):
                pt.data.mul_(1 - cfg.TARGET_TAU).add_(cfg.TARGET_TAU * p.data)
    env.close()
    return q, obs_dim, n_act

In [ ]:
# ---- Roll out the behavior policy (epsilon-greedy) to fill the FIXED dataset. ----
class Dataset:
    "Flat tensors for a fixed offline dataset; supports random minibatch sampling."
    def __init__(self, S, A, R, S2, D):
        self.S  = torch.as_tensor(S,  dtype=torch.float32)
        self.A  = torch.as_tensor(A,  dtype=torch.long)
        self.R  = torch.as_tensor(R,  dtype=torch.float32)
        self.S2 = torch.as_tensor(S2, dtype=torch.float32)
        self.D  = torch.as_tensor(D,  dtype=torch.float32)
        self.n  = len(self.S)
    def sample(self, batch):
        i = torch.randint(0, self.n, (batch,))
        return self.S[i], self.A[i], self.R[i], self.S2[i], self.D[i]

def collect_dataset(cfg, beh_q, seed=0):
    set_seed(seed + 123)
    env = gym.make(cfg.ENV_ID)
    S, A, R, S2, D = [], [], [], [], []
    ep_returns, ep_ret = [], 0.0
    s, _ = env.reset(seed=seed + 123)
    while len(S) < cfg.DATASET_SIZE:
        if random.random() < cfg.EPS_COLLECT:          # epsilon-random for coverage
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = int(beh_q(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        S.append(s); A.append(a); R.append(r); S2.append(s2); D.append(float(term))
        ep_ret += r; s = s2
        if done:
            ep_returns.append(ep_ret); ep_ret = 0.0
            s, _ = env.reset()
    env.close()
    ds = Dataset(np.array(S), np.array(A), np.array(R), np.array(S2), np.array(D))
    return ds, np.array(ep_returns)

In [ ]:
# ---- Run Phase 1 (uses the env; this is the ONLY learning-time env use). ----
t0 = time.time()
beh_q, OBS_DIM, N_ACT = train_behavior_policy(cfg, seed=0)
DATASET, DATA_EP_RETURNS = collect_dataset(cfg, beh_q, seed=0)
BEHAVIOR_RETURN = float(np.mean(DATA_EP_RETURNS))   # reference line for plots later
print(f"Phase 1 done in {time.time()-t0:5.1f}s  "
      f"| dataset = {DATASET.n} transitions, {len(DATA_EP_RETURNS)} episodes")
print(f"Dataset episode return:  mean={DATA_EP_RETURNS.mean():.1f}  "
      f"std={DATA_EP_RETURNS.std():.1f}  "
      f"min={DATA_EP_RETURNS.min():.0f}  max={DATA_EP_RETURNS.max():.0f}")
print(f"[elapsed {time.time()-NB_T0:.1f}s]")

### The dataset is mediocre and mixed — exactly what offline RL must cope with

The histogram below shows the **episode-return distribution of the behavior data**.
It should be *mediocre and spread out* (a "medium" policy diluted by 30% random
actions), not expert. The dashed line is the mean — our **behavior baseline**.
A good offline method should be able to **stitch** the good parts of these
trajectories into a policy that *beats* this mean.

In [ ]:
plt.figure(figsize=(7, 3.2))
plt.hist(DATA_EP_RETURNS, bins=30, color="#6699cc", edgecolor="white")
plt.axvline(BEHAVIOR_RETURN, color="crimson", ls="--", lw=2,
            label=f"behavior mean = {BEHAVIOR_RETURN:.1f}")
plt.title("Phase 1 dataset: episode-return distribution (mediocre / mixed)")
plt.xlabel("episode return"); plt.ylabel("# episodes"); plt.legend()
plt.tight_layout(); plt.show()

## 4. Ground-truth: true discounted returns on the dataset

To later expose Q-overestimation we need a **trusted yardstick**: the *actual*
Monte-Carlo discounted return $\sum_t \gamma^t r_t$ that the behavior policy
realized from each dataset state. We compute these once by walking the stored
transitions backwards within episodes. A well-calibrated $Q(s, a_{\text{data}})$
should sit *near* these returns; naive DQN will float *far above* them.

In [ ]:
def mc_discounted_returns(ds, gamma):
    "Monte-Carlo return-to-go for each transition, computed backwards per episode."
    n = ds.n
    G = np.zeros(n, dtype=np.float32)
    R = ds.R.numpy(); Dn = ds.D.numpy()
    running = 0.0
    for i in range(n - 1, -1, -1):
        # 'done' marks a true terminal; episode boundaries also occur on the
        # last stored index. Reset the accumulator at boundaries.
        if Dn[i] > 0.5:
            running = 0.0
        running = R[i] + gamma * running
        G[i] = running
    return G

MC_RETURNS = mc_discounted_returns(DATASET, cfg.GAMMA)
print(f"MC return-to-go over dataset states: mean={MC_RETURNS.mean():.2f} "
      f"min={MC_RETURNS.min():.2f} max={MC_RETURNS.max():.2f}")

## 5. The four offline learners

Each learner below is **self-contained and minimal (~15-35 lines)** and reads
**only** from the frozen `DATASET`. None calls `env.step()`. Each returns an
object with an `.act(obs)->int` method so eval is uniform.

### 5.1 Behavior Cloning (BC)

The simplest baseline: **supervised classification** of dataset actions.
Minimize cross-entropy between the policy logits and the data action,
$\min_\theta \mathbb{E}_{(s,a)\sim\mathcal D}\big[-\log \pi_\theta(a\mid s)\big]$.
BC never reasons about reward or value, so it **cannot beat the behavior policy** —
it can only reproduce its average. But it is *immune to distributional shift*
because it never queries an OOD action. It is the offline-RL analogue of **SFT**.

In [ ]:
def train_bc(ds, cfg, seed=0):
    set_seed(seed)
    pi  = PolicyNet(OBS_DIM, N_ACT, cfg.HIDDEN)
    opt = torch.optim.Adam(pi.parameters(), lr=cfg.LR)
    last = 0.0
    for step in range(cfg.OFFLINE_STEPS):
        S, A, _, _, _ = ds.sample(cfg.BATCH)
        logits = pi(S)                                   # [B, n_act]
        loss = F.cross_entropy(logits, A)                # supervised: imitate data action
        opt.zero_grad(); loss.backward(); opt.step()
        last = float(loss.item())
    return pi, {"loss": last}

### 5.2 Naive offline DQN — the cautionary tale

Plain DQN run on the fixed buffer. The Bellman target
$y = r + \gamma \max_{a'} Q_{\bar\theta}(s', a')$ queries the **max over all
actions** at $s'$ — including OOD ones. With no environment to refute optimistic
estimates, those errors **bootstrap and compound**: $Q$ drifts *above* the true
returns and the greedy policy chases phantoms. We will see this both in eval
return and in the overestimation plot. We wrap the trained $Q$ in a thin greedy
actor.

In [ ]:
class GreedyQActor:
    "Wrap a Q-network as a greedy policy with a uniform .act(obs) interface."
    def __init__(self, q): self.q = q
    @torch.no_grad()
    def act(self, s_np):
        s = torch.as_tensor(s_np, dtype=torch.float32).unsqueeze(0)
        return int(self.q(s).argmax(dim=-1).item())

def train_naive_dqn(ds, cfg, seed=0):
    set_seed(seed)
    q  = QNet(OBS_DIM, N_ACT, cfg.HIDDEN)
    qt = QNet(OBS_DIM, N_ACT, cfg.HIDDEN); qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=cfg.LR)
    last = 0.0
    for step in range(cfg.OFFLINE_STEPS):
        S, A, R, S2, D = ds.sample(cfg.BATCH)
        with torch.no_grad():
            tgt = R + cfg.GAMMA * (1 - D) * qt(S2).max(dim=1).values   # <-- OOD max
        qsa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(qsa, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        for p, pt in zip(q.parameters(), qt.parameters()):            # soft target
            pt.data.mul_(1 - cfg.TARGET_TAU).add_(cfg.TARGET_TAU * p.data)
        last = float(loss.item())
    return GreedyQActor(q), {"loss": last, "qnet": q}

### 5.3 Conservative Q-Learning (CQL)

CQL keeps the DQN Bellman loss but **adds a conservative penalty** that pushes
$Q$ *down* on all actions and *up* on the dataset action. For discrete actions:

$$
\mathcal{L}_{\text{CQL}}
= \underbrace{\tfrac12\big(Q(s,a) - y\big)^2}_{\text{Bellman}}
\;+\; \alpha\,\underbrace{\Big(\log\!\textstyle\sum_{a'} e^{Q(s,a')}\;-\;Q(s,a_{\text{data}})\Big)}_{\text{conservative gap}}.
$$

The `logsumexp` term is a soft-max over **all** actions (raised by any
over-optimism), while $-Q(s,a_{\text{data}})$ pulls the **data** action back up.
The penalty is large exactly when $Q$ is high on a *non-data* (OOD) action — so it
**caps overestimation** where it would otherwise run away. This is the direct
analogue of a **KL-to-data / reference penalty** in RLHF/DPO.

In [ ]:
def train_cql(ds, cfg, seed=0):
    set_seed(seed)
    q  = QNet(OBS_DIM, N_ACT, cfg.HIDDEN)
    qt = QNet(OBS_DIM, N_ACT, cfg.HIDDEN); qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=cfg.LR)
    last = {}
    for step in range(cfg.OFFLINE_STEPS):
        S, A, R, S2, D = ds.sample(cfg.BATCH)
        qall = q(S)                                              # [B, n_act]
        qsa  = qall.gather(1, A.unsqueeze(1)).squeeze(1)         # Q(s, a_data)
        with torch.no_grad():
            tgt = R + cfg.GAMMA * (1 - D) * qt(S2).max(dim=1).values
        bellman = F.smooth_l1_loss(qsa, tgt)
        # --- conservative penalty: push down logsumexp_a Q(s,a), push up Q(s,a_data) ---
        cql_gap = (torch.logsumexp(qall, dim=1) - qsa).mean()
        loss = bellman + cfg.CQL_ALPHA * cql_gap
        opt.zero_grad(); loss.backward(); opt.step()
        for p, pt in zip(q.parameters(), qt.parameters()):
            pt.data.mul_(1 - cfg.TARGET_TAU).add_(cfg.TARGET_TAU * p.data)
        last = {"loss": float(loss.item()), "bellman": float(bellman.item()),
                "cql_gap": float(cql_gap.item())}
    return GreedyQActor(q), {**last, "qnet": q}

### 5.4 Implicit Q-Learning (IQL)

IQL **never queries $Q$ at OOD actions** — it sidesteps distributional shift by
construction. Three pieces, trained jointly:

1. **Value via expectile regression.** Fit $V(s)$ to the *upper* $\tau$-expectile
   (we use $\tau=0.8$) of $Q(s,a_{\text{data}})$ using the asymmetric loss
   $L_2^\tau(u)=|\tau - \mathbb 1[u<0]|\,u^2$. The upper expectile makes $V$ track
   the **better** in-data actions — an *implicit* max that only ever looks at data.
2. **Q regression.** $Q(s,a)\to r+\gamma V(s')$. The target uses $V(s')$, **not**
   $\max_{a'}Q(s',a')$ — so no OOD action is ever evaluated.
3. **Policy via advantage-weighted BC.** Extract the actor by weighting BC by
   $\exp\!\big(\beta\,(Q(s,a)-V(s))\big)$ (clipped): imitate data actions, but
   *up-weight* the ones with high advantage. This is **exactly** the
   advantage-weighted-regression / DPO-style "stay close to data, lean toward
   high reward" trick.

In [ ]:
def expectile_loss(diff, tau):
    "Asymmetric L2: weight |tau - 1[diff<0]| on diff^2. tau>0.5 => upper expectile."
    w = torch.where(diff > 0, tau, 1.0 - tau)
    return (w * diff.pow(2)).mean()

def train_iql(ds, cfg, seed=0):
    set_seed(seed)
    q   = QNet(OBS_DIM, N_ACT, cfg.HIDDEN)
    qt  = QNet(OBS_DIM, N_ACT, cfg.HIDDEN); qt.load_state_dict(q.state_dict())
    v   = VNet(OBS_DIM, cfg.HIDDEN)
    pi  = PolicyNet(OBS_DIM, N_ACT, cfg.HIDDEN)
    opt = torch.optim.Adam(list(q.parameters()) + list(v.parameters())
                           + list(pi.parameters()), lr=cfg.LR)
    last = {}
    for step in range(cfg.OFFLINE_STEPS):
        S, A, R, S2, D = ds.sample(cfg.BATCH)
        # (1) V: expectile regression toward Q(s, a_data) (use target Q, no grad to it)
        with torch.no_grad():
            q_data = qt(S).gather(1, A.unsqueeze(1)).squeeze(1)
        v_s   = v(S)
        v_loss = expectile_loss(q_data - v_s, cfg.IQL_EXPECTILE)
        # (2) Q: regress to r + gamma * V(s')  -- V(s'), never max_a' Q(s',a')
        with torch.no_grad():
            tgt = R + cfg.GAMMA * (1 - D) * v(S2)
        qsa   = q(S).gather(1, A.unsqueeze(1)).squeeze(1)
        q_loss = F.mse_loss(qsa, tgt)
        # (3) policy: advantage-weighted BC, weight = exp(beta * (Q - V)), clipped
        with torch.no_grad():
            adv = qt(S).gather(1, A.unsqueeze(1)).squeeze(1) - v(S)
            w   = torch.clamp(torch.exp(cfg.IQL_BETA * adv), max=cfg.IQL_ADV_CLIP)
        logp   = F.log_softmax(pi(S), dim=1).gather(1, A.unsqueeze(1)).squeeze(1)
        pi_loss = -(w * logp).mean()
        loss = v_loss + q_loss + pi_loss
        opt.zero_grad(); loss.backward(); opt.step()
        for p, pt in zip(q.parameters(), qt.parameters()):
            pt.data.mul_(1 - cfg.TARGET_TAU).add_(cfg.TARGET_TAU * p.data)
        last = {"v_loss": float(v_loss.item()), "q_loss": float(q_loss.item()),
                "pi_loss": float(pi_loss.item())}
    return pi, {**last, "qnet": q}

## 6. Evaluation (env used only for rollouts)

A uniform rollout harness: run each learned policy *greedily* for
`EVAL_EPISODES` episodes and report the mean return. This is the **only** place
the environment reappears, and only to *score* policies — never to train them.

In [ ]:
@torch.no_grad()
def evaluate(actor, cfg, n_episodes, seed=0):
    "Greedy rollouts; returns array of episode returns."
    env = gym.make(cfg.ENV_ID)
    rets = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        done, ret = False, 0.0
        while not done:
            a = actor.act(s)
            s, r, term, trunc, _ = env.step(a)
            ret += r; done = term or trunc
        rets.append(ret)
    env.close()
    return np.array(rets)

In [ ]:
# ---- Train all four methods across SEEDS and evaluate each. ----
LEARNERS = {
    "BC":         train_bc,
    "Naive DQN":  train_naive_dqn,
    "CQL":        train_cql,
    "IQL":        train_iql,
}

results   = {name: [] for name in LEARNERS}   # name -> list of mean eval returns (per seed)
qnets     = {}                                # name -> a representative Q-net (seed 0) for the Q plot
info_log  = {}

t0 = time.time()
for name, fn in LEARNERS.items():
    for si, seed in enumerate(cfg.SEEDS):
        actor, info = fn(DATASET, cfg, seed=seed)
        ev = evaluate(actor, cfg, cfg.EVAL_EPISODES, seed=100 + seed)
        results[name].append(float(ev.mean()))
        if si == 0:
            info_log[name] = info
            if "qnet" in info:
                qnets[name] = info["qnet"]
    print(f"{name:10s}  eval return: "
          f"{np.mean(results[name]):6.1f} +/- {np.std(results[name]):5.1f}   "
          f"(seeds={cfg.SEEDS})")
print(f"\nPhase 2 (all learners x seeds) done in {time.time()-t0:.1f}s")
print(f"[elapsed {time.time()-NB_T0:.1f}s]")

## 7. Results

### 7.1 Eval return per method (mean ± std over seeds)

The dashed line is the **behavior baseline** (dataset mean return). Read it as:
**BC ≈ behavior** (it can only imitate); **Naive DQN** typically **collapses
below** the data (overestimation → bad greedy policy); **CQL and IQL** should
**meet or beat** the baseline by exploiting reward information *conservatively*.
(Exact numbers vary with the CPU-shrunk budget, but the *ordering* is the lesson.)

In [ ]:
names = list(LEARNERS.keys())
means = [np.mean(results[n]) for n in names]
stds  = [np.std(results[n])  for n in names]
colors = ["#88bb88", "#cc6666", "#6699cc", "#cc9944"]

plt.figure(figsize=(7, 3.8))
plt.bar(names, means, yerr=stds, capsize=6, color=colors, edgecolor="black")
plt.axhline(BEHAVIOR_RETURN, color="crimson", ls="--", lw=2,
            label=f"behavior baseline = {BEHAVIOR_RETURN:.1f}")
plt.ylabel("mean eval return"); plt.title("Offline RL on CartPole: eval return by method")
plt.legend(); plt.tight_layout(); plt.show()

### 7.2 The overestimation diagnostic: naive DQN vs CQL

Here is the heart of the matter. For dataset transitions we compare the model's
**predicted** $Q(s, a_{\text{data}})$ against the **true** Monte-Carlo discounted
return from those states (Section 4). A *calibrated* value function lies near the
diagonal / near the true mean; **naive DQN floats well above it** (overestimation),
while **CQL is pulled back down** toward the truth by its conservative penalty.

In [ ]:
def mean_pred_q_on_data(qnet, ds, n=4000):
    "Mean predicted Q(s, a_data) over a sample of dataset transitions."
    idx = torch.randint(0, ds.n, (min(n, ds.n),))
    with torch.no_grad():
        q = qnet(ds.S[idx]).gather(1, ds.A[idx].unsqueeze(1)).squeeze(1)
    return q.numpy(), idx.numpy()

true_mean = float(MC_RETURNS.mean())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

# (a) bar: mean predicted Q vs true MC return
bar_names, bar_vals = ["true MC\nreturn"], [true_mean]
for n in ["Naive DQN", "CQL"]:
    if n in qnets:
        qv, _ = mean_pred_q_on_data(qnets[n], DATASET)
        bar_names.append(f"{n}\nmean Q"); bar_vals.append(float(qv.mean()))
bcol = ["#444444"] + ["#cc6666", "#6699cc"][:len(bar_vals) - 1]
ax[0].bar(bar_names, bar_vals, color=bcol, edgecolor="black")
ax[0].axhline(true_mean, color="#444444", ls="--", lw=1.5)
ax[0].set_ylabel("value"); ax[0].set_title("Mean predicted Q(s,a_data) vs true return\n(higher than truth = overestimation)")

# (b) scatter: predicted Q vs true return-to-go (calibration)
for n, c in [("Naive DQN", "#cc6666"), ("CQL", "#6699cc")]:
    if n in qnets:
        qv, idx = mean_pred_q_on_data(qnets[n], DATASET, n=1500)
        ax[1].scatter(MC_RETURNS[idx], qv, s=6, alpha=0.3, color=c, label=n)
lim = [min(MC_RETURNS.min(), 0), MC_RETURNS.max() * 1.1 + 1]
ax[1].plot(lim, lim, "k--", lw=1, label="perfect calibration")
ax[1].set_xlabel("true MC return-to-go"); ax[1].set_ylabel("predicted Q(s, a_data)")
ax[1].set_title("Calibration: predicted Q vs true return"); ax[1].legend(markerscale=2)
plt.tight_layout(); plt.show()

print(f"true mean MC return = {true_mean:.2f}")
for n in ["Naive DQN", "CQL"]:
    if n in qnets:
        qv, _ = mean_pred_q_on_data(qnets[n], DATASET)
        print(f"{n:10s} mean predicted Q = {qv.mean():.2f}  "
              f"(gap above truth = {qv.mean()-true_mean:+.2f})")

## 8. Takeaways

* **Distributional shift is the core obstacle of offline RL.** Without
  environment feedback, bootstrapping $\max_{a'}Q(s',a')$ over **OOD actions**
  lets value estimates run away — *naive DQN overestimates and its greedy policy
  collapses*, often **below** the very data it learned from.
* **BC is safe but capped.** It never queries OOD actions, so it is immune to the
  shift — but it can only **reproduce** the behavior policy, never improve on it.
* **CQL fixes it by penalizing.** Adding
  $\alpha(\log\sum_a e^{Q(s,a)} - Q(s,a_{\text{data}}))$ pushes value *down* on
  all actions and *up* on data actions, **capping overestimation** while still
  using reward — so it can **beat** the behavior baseline.
* **IQL fixes it by avoiding.** Expectile-$V$ + $Q\!\to\!r+\gamma V(s')$ +
  advantage-weighted BC means **no OOD action is ever evaluated**, giving stable
  improvement without an explicit penalty.

### Why an AI/ML researcher should care beyond CartPole

* **LLM offline post-training is offline RL.** SFT = behavior cloning;
  RLHF/DPO add a **KL-to-reference** term that plays CQL's conservative role —
  keeping the policy near the data distribution to avoid reward-hacking on OOD
  completions. IQL's advantage-weighted BC is the same family as
  reward-weighted / DPO-style updates.
* **Constrained scientific design** (molecules, sequences, proteins) is offline RL
  against a **fixed learned scorer/oracle**. Optimizing it naively pushes designs
  OOD where the oracle is unreliable — i.e. adversarial, unsynthesizable hits.
  Conservatism / staying-on-manifold is the same cure as CQL/IQL.

See the companion Notion page:
[Offline RL](https://app.notion.com/p/37f95008d766817283ffd9640fdb4c37).

In [ ]:
print(f"Notebook total elapsed: {time.time()-NB_T0:.1f}s")
print("Done. Four offline policies learned from a single fixed dataset, "
      "zero env interaction during learning.")